# GraphRAG — Person A Notebook
## Your job: deliver `knowledge_graph.gml` and chunk JSON files

Run every cell top to bottom. By the end you will have:
- `data/processed/*_chunks.json` — chunks for Person B's vector store
- `outputs/graphs/knowledge_graph.gml` — the knowledge graph for Person B

**Corpus:** Download these 5 PDFs before starting:
1. https://arxiv.org/pdf/1706.03762 — Attention is All You Need
2. https://arxiv.org/pdf/1810.04805 — BERT
3. https://arxiv.org/pdf/2005.14165 — GPT-3
4. https://arxiv.org/pdf/2005.11401 — RAG
5. https://arxiv.org/pdf/2404.16130 — GraphRAG

Place all 5 PDFs in `MyDrive/GraphRAG/data/raw/`

In [ ]:
# ── Cell 1: Mount Drive + setup ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/GraphRAG'

if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(f'Project not found at {PROJECT_ROOT}')

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Create required directories
for d in ['data/raw', 'data/processed', 'outputs/graphs',
          'cache/extractions', 'outputs/summaries']:
    os.makedirs(d, exist_ok=True)

print('Working directory:', os.getcwd())
print('PDFs in data/raw:', os.listdir('data/raw'))
print('\n✅ Cell 1 passed')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────
import subprocess, sys

def install(packages, label):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
        capture_output=True, text=True
    )
    print(f"{'✅' if result.returncode == 0 else '❌'} {label}")
    if result.returncode != 0:
        print(result.stderr[-400:])

install(['pyyaml', 'python-dotenv'], 'Core utilities')
install(['pymupdf'],                 'PyMuPDF (PDF parser)')
install(['tiktoken'],                'Tokenizer')
install(['spacy'],                   'spaCy')
install(['groq'],                    'Groq API')
install(['google-generativeai'],     'Gemini API')
install(['networkx'],                'NetworkX')

r = subprocess.run(
    [sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
    capture_output=True, text=True
)
print('✅ spaCy model' if r.returncode == 0 else '❌ spaCy: ' + r.stderr[-200:])
print('\n--- Done ---')

In [ ]:
# ── Cell 3: Set API keys ──────────────────────────────────────
# Get free Groq key at: console.groq.com → API Keys → Create
# Use Groq — Gemini free tier has quota issues

GROQ_API_KEY = 'gsk_paste_your_groq_key_here'

if GROQ_API_KEY == 'gsk_paste_your_groq_key_here':
    raise ValueError('Paste your Groq API key above before running.')

import os
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# Write .env file
with open('.env', 'w') as f:
    f.write(f'GROQ_API_KEY={GROQ_API_KEY}\nGEMINI_API_KEY=none\n')

# Override config to use Groq
from src.config import cfg
cfg['llm']['extraction_provider'] = 'groq'
cfg['llm']['extraction_model']    = 'llama-3.3-70b-versatile'
cfg['llm']['generation_provider'] = 'groq'
cfg['llm']['generation_model']    = 'llama-3.3-70b-versatile'

print('Provider: Groq — llama-3.3-70b-versatile')
print('\n✅ Cell 3 passed')

In [ ]:
# ── Cell 4: PDF extractor ─────────────────────────────────────
# Reads every PDF in data/raw/ and saves clean text to data/raw/

import fitz  # PyMuPDF
import json
from pathlib import Path

def extract_pdf(pdf_path: str) -> str:
    """Extract clean text from a PDF file."""
    doc  = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text()
        # Basic cleaning
        text = text.replace('\x00', '')   # null bytes
        text = '\n'.join(
            line for line in text.split('\n')
            if len(line.strip()) > 20      # remove header/footer noise
        )
        pages.append(text)
    return '\n\n'.join(pages)

raw_dir = Path('data/raw')
pdf_files = list(raw_dir.glob('*.pdf'))

if not pdf_files:
    raise FileNotFoundError(
        'No PDFs found in data/raw/\n'
        'Upload your 5 arXiv papers there first.'
    )

documents = []
for pdf_path in pdf_files:
    doc_id = pdf_path.stem.replace(' ', '_').lower()
    text   = extract_pdf(str(pdf_path))
    documents.append({'doc_id': doc_id, 'text': text})

    # Save extracted text
    out_path = raw_dir / f'{doc_id}_text.txt'
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)

    print(f'  {doc_id}: {len(text):,} characters extracted')

print(f'\n{len(documents)} documents extracted')
print('\n✅ Cell 4 passed — PDFs extracted')

In [ ]:
# ── Cell 5: Chunk all documents ───────────────────────────────
# Uses Person B's chunker — produces JSON files Person B's vector store needs

from src.chunker import chunk_all_documents, load_all_chunks

all_chunks = chunk_all_documents(documents)

print(f'\nChunking summary:')
from collections import Counter
per_doc = Counter(c['doc_id'] for c in all_chunks)
for doc_id, count in per_doc.items():
    print(f'  {doc_id}: {count} chunks')

print(f'\nTotal chunks: {len(all_chunks)}')
print('Saved to: data/processed/')

assert len(all_chunks) > 0, 'No chunks produced — check PDF extraction'
print('\n✅ Cell 5 passed — all documents chunked')

In [ ]:
# ── Cell 6: Test LLM before running extraction ────────────────
# Always test the LLM works BEFORE running on all chunks
# Extraction is expensive — don't discover the key is broken after 2 hours

from src.llm_client import LLMClient

llm = LLMClient(purpose='extraction')
print('Provider:', llm.provider, '| Model:', llm.model)

test = llm.generate('In one sentence, what is the Transformer architecture?')
print('Test response:', test)

assert len(test) > 20, 'Response too short — check API key'
print('\n✅ Cell 6 passed — LLM ready for extraction')

In [ ]:
# ── Cell 7: Define extraction prompts ─────────────────────────
# These prompts are the most important thing in the entire project.
# Quality of extraction directly determines quality of the knowledge graph.

EXTRACTION_PROMPT = """You are an expert knowledge graph builder analyzing academic NLP/AI research papers.

Given the following text chunk from a research paper, extract:
1. Important ENTITIES (models, algorithms, techniques, concepts, datasets, metrics)
2. RELATIONSHIPS between those entities

TEXT CHUNK:
{chunk_text}

Return ONLY valid JSON in this exact format, no explanation:
{{
  "entities": [
    {{"name": "entity name", "type": "model|algorithm|technique|concept|dataset|metric", "description": "one sentence description"}}
  ],
  "relationships": [
    {{"source": "entity1", "relation": "relation_type", "target": "entity2", "description": "brief description", "weight": 7}}
  ]
}}

Rules:
- Extract 3-10 entities per chunk
- Extract 3-10 relationships per chunk
- Use specific relation types: based_on, uses, extends, improves_on, trained_with, evaluated_on, part_of, enables, introduces, compared_to
- Weight is 1-10 (10 = strongest/most direct relationship)
- Only extract entities clearly mentioned in this chunk
- Use canonical names (BERT not bert, Transformer not transformer)"""

GLEANING_PROMPT = """You previously extracted entities and relationships from a text chunk.
Here is what you found:
{previous_extraction}

Original text:
{chunk_text}

Are there any important entities or relationships you MISSED?
Return ONLY a JSON object with any additional findings (same format as before).
If nothing was missed, return: {{"entities": [], "relationships": []}}"""

print('Extraction prompt: ready')
print('Gleaning prompt: ready')
print('\n✅ Cell 7 passed — prompts defined')

In [ ]:
# ── Cell 8: Test extraction on 3 chunks ───────────────────────
# ALWAYS test on a small sample before running on all chunks
# Inspect the output manually — if entities look wrong, fix the prompt NOW

import json, hashlib, time

def parse_llm_json(text: str) -> dict:
    """Parse JSON from LLM output, handling markdown fences."""
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    return json.loads(text)

def extract_chunk(chunk: dict, llm, glean_rounds: int = 2) -> dict:
    """Extract entities and relations from one chunk with gleaning."""
    # Check cache first
    cache_key  = hashlib.md5(chunk['text'].encode()).hexdigest()[:12]
    cache_path = f"cache/extractions/{cache_key}.json"

    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)

    # Initial extraction
    prompt   = EXTRACTION_PROMPT.format(chunk_text=chunk['text'][:3000])
    raw      = llm.generate(prompt, temperature=0.0)
    result   = parse_llm_json(raw)
    entities = result.get('entities', [])
    rels     = result.get('relationships', [])

    # Gleaning rounds
    for round_num in range(glean_rounds):
        time.sleep(1.0)  # rate limit safety
        glean_prompt = GLEANING_PROMPT.format(
            previous_extraction=json.dumps(result, indent=2)[:1500],
            chunk_text=chunk['text'][:2000]
        )
        glean_raw    = llm.generate(glean_prompt, temperature=0.0)
        glean_result = parse_llm_json(glean_raw)
        entities    += glean_result.get('entities', [])
        rels        += glean_result.get('relationships', [])

    output = {
        'chunk_id':     chunk['chunk_id'],
        'doc_id':       chunk['doc_id'],
        'entities':     entities,
        'relationships': rels,
    }

    # Cache result
    with open(cache_path, 'w') as f:
        json.dump(output, f, indent=2)

    return output

# Test on first 3 chunks only
test_results = []
for chunk in all_chunks[:3]:
    print(f"Testing chunk: {chunk['chunk_id']}")
    result = extract_chunk(chunk, llm, glean_rounds=1)  # 1 round for test
    test_results.append(result)
    print(f"  Entities: {len(result['entities'])}")
    for e in result['entities'][:3]:
        print(f"    - {e['name']} ({e['type']}): {e['description'][:60]}")
    print(f"  Relations: {len(result['relationships'])}")
    for r in result['relationships'][:3]:
        print(f"    - {r['source']} --[{r['relation']}]--> {r['target']}")
    print()
    time.sleep(1.5)

print('--- Manual check ---')
print('Do the entities look correct? Are they real NLP concepts?')
print('Are the relationships meaningful?')
print('If YES → run Cell 9')
print('If NO  → go back to Cell 7 and improve the prompt')
print('\n✅ Cell 8 passed — test extraction complete')

In [ ]:
# ── Cell 9: Full extraction on all chunks ─────────────────────
# WARNING: This will make many API calls.
# With 5 papers (~200 chunks total) + 2 gleaning rounds = ~600 API calls
# At Groq free tier: takes ~30-45 minutes
# SAFE TO INTERRUPT AND RESTART — all results cached to disk

import time
from tqdm.notebook import tqdm

all_extractions = []
skipped = 0
failed  = []

print(f'Processing {len(all_chunks)} chunks...')
print('Results cached after each chunk — safe to interrupt\n')

for i, chunk in enumerate(tqdm(all_chunks)):
    cache_key  = hashlib.md5(chunk['text'].encode()).hexdigest()[:12]
    cache_path = f"cache/extractions/{cache_key}.json"

    if os.path.exists(cache_path):
        with open(cache_path) as f:
            all_extractions.append(json.load(f))
        skipped += 1
        continue

    try:
        result = extract_chunk(chunk, llm, glean_rounds=2)
        all_extractions.append(result)
        time.sleep(1.5)  # rate limit delay
    except Exception as e:
        print(f'\n⚠️  Failed chunk {chunk["chunk_id"]}: {e}')
        failed.append(chunk['chunk_id'])
        time.sleep(3.0)  # longer wait after failure
        continue

print(f'\nExtraction complete:')
print(f'  Processed: {len(all_extractions) - skipped}')
print(f'  From cache: {skipped}')
print(f'  Failed: {len(failed)}')
if failed:
    print(f'  Failed chunks: {failed[:5]}')

total_entities = sum(len(e['entities']) for e in all_extractions)
total_rels     = sum(len(e['relationships']) for e in all_extractions)
print(f'  Total entities extracted: {total_entities}')
print(f'  Total relationships extracted: {total_rels}')

# Save all extractions to one file
with open('cache/all_extractions.json', 'w') as f:
    json.dump(all_extractions, f, indent=2)

print('\n✅ Cell 9 passed — full extraction complete')

In [ ]:
# ── Cell 10: Entity resolution ────────────────────────────────
# Merge duplicate entities across all chunks
# e.g. "BERT", "the BERT model", "bert" → one canonical node

from difflib import SequenceMatcher
from collections import defaultdict

def normalize(name: str) -> str:
    return name.lower().strip().replace('-', ' ').replace('_', ' ')

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

# Collect all entities across all chunks
raw_entities = defaultdict(list)  # normalized_name → list of raw names

for extraction in all_extractions:
    for entity in extraction['entities']:
        name = entity['name'].strip()
        if not name or len(name) < 2:
            continue
        norm = normalize(name)
        raw_entities[norm].append({
            'raw_name':   name,
            'type':       entity.get('type', 'concept'),
            'description': entity.get('description', ''),
        })

print(f'Unique normalized entity names: {len(raw_entities)}')

# Build canonical entity map
# Strategy: pick the most frequent raw name as canonical
canonical_map = {}  # raw_name → canonical_name
entity_data   = {}  # canonical_name → merged data

for norm_name, occurrences in raw_entities.items():
    # Most common raw name = canonical
    from collections import Counter
    name_counts = Counter(o['raw_name'] for o in occurrences)
    canonical   = name_counts.most_common(1)[0][0]

    # Merge descriptions
    descriptions = list(set(
        o['description'] for o in occurrences if o['description']
    ))
    merged_desc = ' '.join(descriptions[:2])[:300]  # cap length

    # Most common type
    type_counts = Counter(o['type'] for o in occurrences)
    entity_type = type_counts.most_common(1)[0][0]

    entity_data[canonical] = {
        'description': merged_desc,
        'entity_type': entity_type,
        'frequency':   len(occurrences),
    }

    for occ in occurrences:
        canonical_map[occ['raw_name']] = canonical

print(f'Canonical entities after resolution: {len(entity_data)}')
print('\nTop 10 by frequency:')
top = sorted(entity_data.items(), key=lambda x: x[1]['frequency'], reverse=True)[:10]
for name, data in top:
    print(f'  {data["frequency"]:3d}x  {name} ({data["entity_type"]})')

print('\n✅ Cell 10 passed — entity resolution complete')

In [ ]:
# ── Cell 11: Build knowledge graph ───────────────────────────
# Assemble the NetworkX MultiDiGraph from resolved entities + relations

import networkx as nx

G = nx.MultiDiGraph()

# Add nodes
for name, data in entity_data.items():
    G.add_node(
        name,
        entity_type=data['entity_type'],
        description=data['description'],
        frequency=data['frequency'],
    )

# Add edges
edge_count  = 0
skip_count  = 0

for extraction in all_extractions:
    for rel in extraction['relationships']:
        src_raw = rel.get('source', '').strip()
        tgt_raw = rel.get('target', '').strip()

        # Resolve to canonical names
        src = canonical_map.get(src_raw, src_raw)
        tgt = canonical_map.get(tgt_raw, tgt_raw)

        # Skip if either node not in graph
        if src not in G.nodes or tgt not in G.nodes:
            skip_count += 1
            continue

        # Skip self-loops
        if src == tgt:
            continue

        weight = float(rel.get('weight', 5))
        G.add_edge(
            src, tgt,
            relation_type=rel.get('relation', 'related_to'),
            description=rel.get('description', ''),
            weight=weight,
            computed_weight=weight,
        )
        edge_count += 1

print(f'Graph assembled:')
print(f'  Nodes: {G.number_of_nodes()}')
print(f'  Edges: {G.number_of_edges()}')
print(f'  Edges skipped (unresolved nodes): {skip_count}')

# Graph health check
isolated = [n for n in G.nodes() if G.degree(n) == 0]
print(f'  Isolated nodes (no edges): {len(isolated)}')
if isolated:
    # Remove isolated nodes — they add noise
    G.remove_nodes_from(isolated)
    print(f'  Removed {len(isolated)} isolated nodes')

# Show top entities
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 entities by connections:')
for name, deg in degrees:
    print(f'  {deg:3d}  {name}')

assert G.number_of_nodes() > 20, 'Too few nodes — check extraction quality'
assert G.number_of_edges() > 30, 'Too few edges — check extraction quality'
print('\n✅ Cell 11 passed — knowledge graph built')

In [ ]:
# ── Cell 12: Save graph + validate ───────────────────────────
import os

os.makedirs('outputs/graphs', exist_ok=True)
graph_path = 'outputs/graphs/knowledge_graph.gml'

nx.write_gml(G, graph_path)
print(f'Graph saved → {graph_path}')

# Validate by reloading
G_check = nx.read_gml(graph_path)
print(f'Reload check: {G_check.number_of_nodes()} nodes, {G_check.number_of_edges()} edges')
assert G_check.number_of_nodes() == G.number_of_nodes(), 'Node count mismatch after reload'

# Print sample node to verify attributes saved correctly
sample_node = list(G_check.nodes(data=True))[0]
print(f'\nSample node: {sample_node[0]}')
print(f'  Attributes: {list(sample_node[1].keys())}')

# Print sample edge
sample_edges = list(G_check.edges(data=True))[:1]
if sample_edges:
    u, v, d = sample_edges[0]
    print(f'\nSample edge: {u} --[{d.get("relation_type")}]--> {v}')

print('\n' + '='*50)
print('HANDOFF CHECKLIST FOR PERSON B')
print('='*50)
print(f'\n1. knowledge_graph.gml')
print(f'   Path: {os.path.abspath(graph_path)}')
print(f'   Size: {os.path.getsize(graph_path):,} bytes')
print(f'   Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
print(f'\n2. Chunk JSON files')
chunk_files = list(Path('data/processed').glob('*_chunks.json'))
for f in chunk_files:
    print(f'   {f.name}')
print(f'\nShare both with Person B immediately.')
print('\n✅ Cell 12 passed — graph saved and ready for handoff')

In [ ]:
# ── Cell 13: Quick visualisation check ───────────────────────
# Visual sanity check — do the connected components make sense?

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Plot top 40 nodes by degree
top_nodes = [n for n, _ in sorted(
    G.degree(), key=lambda x: x[1], reverse=True
)[:40]]
subG = G.subgraph(top_nodes)

fig, ax = plt.subplots(figsize=(14, 10))
pos     = nx.spring_layout(subG, seed=42, k=2)

# Size nodes by degree
sizes = [G.degree(n) * 80 for n in subG.nodes()]

nx.draw_networkx_nodes(subG, pos, node_size=sizes,
                       node_color='#4A90D9', alpha=0.8, ax=ax)
nx.draw_networkx_labels(subG, pos, font_size=7,
                        font_color='white', ax=ax)
nx.draw_networkx_edges(subG, pos, alpha=0.3,
                       edge_color='gray',
                       arrows=True, ax=ax)

ax.set_title(f'Knowledge Graph — Top 40 nodes by degree\n'
             f'Total: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges',
             fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('outputs/graphs/kg_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print('Visualization saved → outputs/graphs/kg_visualization.png')
print('\nLook at the graph:')
print('- Are the central nodes (largest circles) the right concepts?')
print('- Do connected clusters look thematically related?')
print('- If it looks like random noise → extraction quality needs improvement')
print('\n✅ Cell 13 — graph visualization complete')

In [ ]:
# ── Cell 14: Final summary ────────────────────────────────────
print('=' * 55)
print('PERSON A — ALL DONE')
print('=' * 55)
print(f"""
Deliverables ready for Person B:

  1. outputs/graphs/knowledge_graph.gml
     → {G.number_of_nodes()} nodes, {G.number_of_edges()} edges

  2. data/processed/*_chunks.json
     → {len(all_chunks)} total chunks from {len(documents)} documents

Share both folders with Person B now.
They need to drop these into their Drive and re-run
cells 10-12 of their smoke test notebook.

Then you both run evaluation together.
""")